In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_20.pickle

In [ ]:
%%cudf.pandas.profile
### cell 20 ###

def add_gap_rows(essay):
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # filter once on GPU
    df = train[train.id == essay][cols_to_keep].reset_index(drop=True)
    # compute previous end (-1 for the first row)
    df['prev_end'] = df.discourse_end.shift(1).fillna(-1).astype('int32')
    # build all gap‐rows in one shot where gap_length > 0
    df_gap = df[df.gap_length > 0]
    new_rows = cudf.DataFrame({
        'discourse_start': df_gap.prev_end + 1,
        'discourse_end':   df_gap.discourse_start - 1,
        'discourse_type': ['Nothing'] * len(df_gap),
        'gap_length':     [None] * len(df_gap),
        'gap_end_length': [None] * len(df_gap)
    })
    # append final gap if needed
    if df.gap_end_length.iloc[-1] > 0:
        last = df.iloc[-1]
        end_row = cudf.DataFrame({
            'discourse_start': [int(last.discourse_end) + 1],
            'discourse_end':   [int(last.discourse_end) + 1 + int(last.gap_end_length)],
            'discourse_type': ['Nothing'],
            'gap_length':     [None],
            'gap_end_length': [None]
        })
        new_rows = cudf.concat([new_rows, end_row], ignore_index=True)
    # combine original and gap rows, then sort
    df_essay = cudf.concat([df[cols_to_keep], new_rows], ignore_index=True)
    df_essay = df_essay.sort_values('discourse_start').reset_index(drop=True)
    return df_essay


def print_colored_essay(essay):
    df_essay = add_gap_rows(essay)
    essay_file = f"/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train/{essay}.txt"
    # vectorize ents construction
    starts = df_essay.discourse_start.to_pandas().tolist()
    ends   = df_essay.discourse_end.to_pandas().tolist()
    labels = df_essay.discourse_type.to_pandas().tolist()
    ents = [
        {'start': int(s), 'end': int(e), 'label': l}
        for s, e, l in zip(starts, ends, labels)
    ]
    with open(essay_file, 'r') as f:
        data = f.read()
    doc2 = {'text': data, 'ents': ents}
    colors = {
        'Lead': '#EE11D0','Position': '#AB4DE1','Claim': '#1EDE71',
        'Evidence': '#33FAFA','Counterclaim': '#4253C1',
        'Concluding Statement': 'yellow','Rebuttal': 'red'
    }
    options = {
        'ents': df_essay.discourse_type.unique().to_pandas().tolist(),
        'colors': colors
    }